In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dlmmdd_workshop_synthetic_source_attribution_challenge_path = kagglehub.competition_download('dlmmdd-workshop-synthetic-source-attribution-challenge')

print('Data source import complete.')


100%|██████████| 10.1G/10.1G [08:50<00:00, 20.4MB/s]

Extracting files...


Data source import complete.


In [ ]:
import os, shutil

ckpt_zip = '/content/drive/MyDrive/dlmmdd_pad_processedval_eva02_large448_96gb_outputs.zip'
if not os.path.exists('/content/dlmmdd_pad_processedval_eva02_large448_96gb_outputs'):
    if os.path.exists(ckpt_zip):
        shutil.copy(ckpt_zip, '/content/')
        shutil.unpack_archive('/content/' + os.path.basename(ckpt_zip),
                             '/content/dlmmdd_pad_processedval_eva02_large448_96gb_outputs', 'zip')
        print('extracted ckpts')
    else:
        raise FileNotFoundError(f'Missing {ckpt_zip} on Drive')
else:
    print('ckpts already extracted')

# Pseudo-label inputs from the 0.997333 ensemble run
for fn in ('ensemble_logprobs.npy', 'ensemble_ids.csv'):
    src = f'/content/drive/MyDrive/{fn}'
    dst = f'/content/{fn}'
    if not os.path.exists(dst):
        if not os.path.exists(src):
            raise FileNotFoundError(f'Missing {src} on Drive')
        shutil.copy(src, dst)
        print(f'copied {fn}')
    else:
        print(f'{fn} already present')

extracted ckpts
copied ensemble_logprobs.npy
copied ensemble_ids.csv


In [ ]:

LOGPROBS_PATH = '/content/drive/MyDrive/ensemble_logprobs.npy'
IDS_CSV_PATH  = '/content/drive/MyDrive/ensemble_ids.csv'   # has an 'ID' column

# Fallback if ensemble_ids.csv is missing (competition test.csv ID order).
# Only used if IDS_CSV_PATH does not exist.
TEST_CSV_FALLBACK = '/root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data/test.csv'

NUM_CLASSES = 10
PER_CLASS   = 300        # the verified test marginal
import os
print('logprobs exists:', os.path.exists(LOGPROBS_PATH))
print('ids csv exists  :', os.path.exists(IDS_CSV_PATH))

logprobs exists: True
ids csv exists  : True


In [ ]:
import numpy as np
import pandas as pd

logp = np.load(LOGPROBS_PATH)            # (N, 10) log-probabilities
assert logp.ndim == 2 and logp.shape[1] == NUM_CLASSES, f'unexpected shape {logp.shape}'
N = logp.shape[0]
print('loaded logprobs:', logp.shape)

# Load IDs (preserve the exact row order the logprobs were saved in).
if os.path.exists(IDS_CSV_PATH):
    ids_df = pd.read_csv(IDS_CSV_PATH)
else:
    print('ensemble_ids.csv not found - falling back to test.csv')
    ids_df = pd.read_csv(TEST_CSV_FALLBACK)
ids = ids_df['ID'].astype(int).to_numpy()
assert len(ids) == N, f'IDs ({len(ids)}) != logprobs rows ({N})'

# Probabilities (logp is log-softmax; exp recovers probs). 
probs = np.exp(logp - logp.max(1, keepdims=True))
probs = probs / probs.sum(1, keepdims=True)

if N % NUM_CLASSES != 0:
    print(f'WARNING: N={N} is not divisible by {NUM_CLASSES}. PER_CLASS={PER_CLASS} may not fit.')
print('rows', N, ' expected per-class', PER_CLASS)

loaded logprobs: (3000, 10)
rows 3000  expected per-class 300


In [ ]:
raw_preds = probs.argmax(1)
counts = np.bincount(raw_preds, minlength=NUM_CLASSES)
print('=== Current predicted class marginal (raw argmax) ===')
dev = counts - PER_CLASS
for c in range(NUM_CLASSES):
    flag = '' if dev[c] == 0 else f'  ({"+" if dev[c]>0 else ""}{dev[c]})'
    print(f'  class {c}: {counts[c]:4d}{flag}')
print(f'  total: {counts.sum()}')

abs_dev = int(np.abs(dev).sum())
min_errors = int(dev[dev > 0].sum())
print(f'\nSum of |deviations| = {abs_dev}')
print(f'>>> LOWER BOUND on misclassified images = {min_errors}')
print('    (each over-predicted slot must be a wrong image; errors that cancel')
print('     within the marginal are invisible, so the true count is >= this.)')

# Mean confidence overall, for context.
print(f'\nMean top-1 confidence: {probs.max(1).mean():.4f}')

=== Current predicted class marginal (raw argmax) ===
  class 0:  302  (+2)
  class 1:  301  (+1)
  class 2:  299  (-1)
  class 3:  299  (-1)
  class 4:  302  (+2)
  class 5:  301  (+1)
  class 6:  299  (-1)
  class 7:  299  (-1)
  class 8:  301  (+1)
  class 9:  297  (-3)
  total: 3000

Sum of |deviations| = 14
>>> LOWER BOUND on misclassified images = 7
    (each over-predicted slot must be a wrong image; errors that cancel
     within the marginal are invisible, so the true count is >= this.)

Mean top-1 confidence: 0.9953


In [ ]:
# METHOD 1: Sinkhorn soft balancing
# Iteratively rescale so columns sum to PER_CLASS and rows sum to 1, then argmax.
# Soft: only images near a decision boundary move.
def sinkhorn_balance(probs, per_class, iters=200, eps=1e-12):
    P = probs.astype(np.float64).copy()
    for _ in range(iters):
        # column constraint: each class total -> per_class
        P *= per_class / (P.sum(0, keepdims=True) + eps)
        # row constraint: each row -> 1
        P /= (P.sum(1, keepdims=True) + eps)
    return P

P_sink = sinkhorn_balance(probs, PER_CLASS)
sink_preds = P_sink.argmax(1)
sink_counts = np.bincount(sink_preds, minlength=NUM_CLASSES)
n_changed_sink = int((sink_preds != raw_preds).sum())
print('Sinkhorn class marginal :', sink_counts.tolist())
print(f'Sinkhorn changed {n_changed_sink} predictions vs raw argmax')
print('(Sinkhorn is soft - marginal will be approximately, not exactly, 300/class.)')

Sinkhorn class marginal : [300, 301, 299, 300, 301, 301, 299, 299, 301, 299]
Sinkhorn changed 3 predictions vs raw argmax
(Sinkhorn is soft - marginal will be approximately, not exactly, 300/class.)


In [ ]:
# METHOD 2: exact balanced assignment
# Assign each image to a class so EVERY class gets exactly PER_CLASS images,
# maximising total log-probability. Solved as a min-cost assignment: expand each
# class into PER_CLASS identical 'slots' -> N x N cost matrix.
from scipy.optimize import linear_sum_assignment

assert N == PER_CLASS * NUM_CLASSES, (
    f'Exact assignment needs N == PER_CLASS*NUM_CLASSES ({PER_CLASS*NUM_CLASSES}); got {N}.')

# cost[i, slot] = -logprob[i, class(slot)]   (minimising cost == maximising logprob)
cost = np.empty((N, N), dtype=np.float64)
for c in range(NUM_CLASSES):
    cost[:, c*PER_CLASS:(c+1)*PER_CLASS] = -logp[:, c:c+1]

print('solving exact balanced assignment (3000x3000)... ~a few seconds')
row_ind, col_ind = linear_sum_assignment(cost)
exact_preds = np.empty(N, dtype=int)
exact_preds[row_ind] = col_ind // PER_CLASS
exact_counts = np.bincount(exact_preds, minlength=NUM_CLASSES)
n_changed_exact = int((exact_preds != raw_preds).sum())
print('Exact class marginal    :', exact_counts.tolist(), '(every class == 300)')
print(f'Exact assignment changed {n_changed_exact} predictions vs raw argmax')

solving exact balanced assignment (3000x3000)... ~a few seconds
Exact class marginal    : [300, 300, 300, 300, 300, 300, 300, 300, 300, 300] (every class == 300)
Exact assignment changed 7 predictions vs raw argmax


In [ ]:
# Inspect which predictions the exact method changed, and how confident they were.
# If it mostly flips LOW-confidence predictions -> safe. If it flips high-confidence
# ones -> the model and the marginal disagree strongly; be cautious.
changed = np.where(exact_preds != raw_preds)[0]
raw_conf = probs.max(1)
print(f'{len(changed)} predictions changed by exact balancing.')
if len(changed):
    cc = raw_conf[changed]
    print(f'  confidence of changed preds: min={cc.min():.3f} med={np.median(cc):.3f} max={cc.max():.3f}')
    print(f'  changed preds with raw confidence > 0.90: {int((cc>0.90).sum())}  (want this LOW)')
    SOURCES = ['AuraFlow','Freepik','Lumina','Photon','Pixart','Playgnd2.5','SD3','SD3.5','SDXL-Turbo','Hunyuan']
    rows = []
    for i in changed:
        rows.append({'ID': int(ids[i]),
                     'raw': f'{raw_preds[i]}:{SOURCES[raw_preds[i]]}',
                     'balanced': f'{exact_preds[i]}:{SOURCES[exact_preds[i]]}',
                     'raw_conf': round(float(raw_conf[i]), 3),
                     'p_balanced': round(float(probs[i, exact_preds[i]]), 3)})
    chg_df = pd.DataFrame(rows).sort_values('raw_conf')
    print('\nAll changed predictions (sorted by raw confidence, least confident first):')
    print(chg_df.to_string(index=False))

7 predictions changed by exact balancing.
  confidence of changed preds: min=0.535 med=0.656 max=0.928
  changed preds with raw confidence > 0.90: 1  (want this LOW)

All changed predictions (sorted by raw confidence, least confident first):
  ID          raw  balanced  raw_conf  p_balanced
4297   0:AuraFlow 9:Hunyuan     0.535       0.419
5960   0:AuraFlow  3:Photon     0.555       0.398
3170     4:Pixart 9:Hunyuan     0.652       0.333
5734     4:Pixart 9:Hunyuan     0.656       0.205
7126 8:SDXL-Turbo     6:SD3     0.789       0.191
1813 5:Playgnd2.5   7:SD3.5     0.834       0.105
8047    1:Freepik  2:Lumina     0.928       0.064


In [ ]:
def write_sub(preds, name):
    sub = pd.DataFrame({'ID': ids.astype(int), 'TARGET': preds.astype(int)})
    path = f'/content/{name}'
    sub.to_csv(path, index=False)
    print('saved', path)
    return path

p_raw   = write_sub(raw_preds,   'submission_raw_argmax.csv')
p_sink  = write_sub(sink_preds,  'submission_balanced_sinkhorn.csv')
p_exact = write_sub(exact_preds, 'submission_balanced_exact.csv')

# Backup to Drive
import shutil
for p in (p_raw, p_sink, p_exact):
    try:
        shutil.copy(p, '/content/drive/MyDrive/')
    except Exception as e:
        print('drive copy skipped:', e)
print('\nDone.')

saved /content/submission_raw_argmax.csv
saved /content/submission_balanced_sinkhorn.csv
saved /content/submission_balanced_exact.csv

Done.
